# Corrected Deep Ritz thermoelastic rod benchmark

This Colab notebook reproduces the corrected manuscript **Variational Deep Ritz Modeling of a Variable-Area Thermoelastic Rod with Finite-Element Verification**.

It contains: **Case A** (fixed-free, end load), **Case B** (fixed-fixed, restrained thermal expansion), and an independent **FEM convergence check**. Run all cells from top to bottom. The code uses float64 arithmetic and seed 42.


In [ ]:
import platform, sys
import numpy as np
import pandas as pd
import matplotlib
import torch

print("Python:", sys.version)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("Matplotlib:", matplotlib.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
import json, platform, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# ============================================================
# Deep Ritz PINN for a 1D thermoelastic rod
# Reproducible Colab/GitHub version
# ============================================================

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_default_dtype(torch.float64)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------- Physical parameters ----------------
L = 1.0                  # m
A0 = 1.0e-3              # m^2
BETA = 3.0
E = 200.0e9              # Pa
ALPHA = 1.2e-5           # 1/K
DELTA_T_MAX = 100.0      # K
P = 1.0e4                # N

# ---------------- Neural network ----------------
HIDDEN_LAYERS = 4
NEURONS = 50

# ---------------- Optimization ----------------
N_QUADRATURE = 200
ADAM_EPOCHS = 5000
ADAM_LR = 1.0e-3
LBFGS_MAX_ITER = 500

# Independent validation grid
N_TEST = 400

OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(exist_ok=True)

# Characteristic displacement for numerical conditioning.
# It does NOT change the physical solution; it only scales the NN output.
U_SCALE = P * L / (E * A0) + ALPHA * DELTA_T_MAX * L


# ============================================================
# Geometry and thermal field
# ============================================================

def area(x):
    return A0 / (1.0 + BETA * (x / L) ** 2)


def temperature_change(x):
    return DELTA_T_MAX * (x / L)


def area_np(x):
    return A0 / (1.0 + BETA * (x / L) ** 2)


def temperature_change_np(x):
    return DELTA_T_MAX * (x / L)


# ============================================================
# Analytical solution
# ============================================================

def analytical_displacement(x):
    x = np.asarray(x, dtype=float)
    return (
        P / (E * A0)
        * (x + BETA * x**3 / (3.0 * L**2))
        + ALPHA * DELTA_T_MAX * x**2 / (2.0 * L)
    )


def analytical_stress(x):
    x = np.asarray(x, dtype=float)
    return P / area_np(x)


# ============================================================
# Deep Ritz network
# ============================================================

class DeepRitzThermoelasticity(nn.Module):
    """
    Dimensionless coordinate and output scaling improve conditioning.

    u_theta(x) = U_SCALE * (x/L) * N_theta(x/L)

    Therefore u_theta(0)=0 exactly.
    """

    def __init__(self, hidden_layers=4, neurons_per_layer=50):
        super().__init__()

        layers = [
            nn.Linear(1, neurons_per_layer),
            nn.Tanh()
        ]

        for _ in range(hidden_layers - 1):
            layers.extend([
                nn.Linear(neurons_per_layer, neurons_per_layer),
                nn.Tanh()
            ])

        layers.append(nn.Linear(neurons_per_layer, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        xi = x / L
        return U_SCALE * xi * self.net(xi)


# ============================================================
# Gauss-Legendre quadrature
# ============================================================

def gauss_legendre_points(n):
    nodes, weights = np.polynomial.legendre.leggauss(n)

    x = 0.5 * L * (nodes + 1.0)
    w = 0.5 * L * weights

    x = torch.tensor(x, device=DEVICE).view(-1, 1)
    w = torch.tensor(w, device=DEVICE).view(-1, 1)

    return x, w


# ============================================================
# Dimensionless Deep Ritz energy
# ============================================================

def compute_energy_loss(model, x_quad, weights):
    """
    Physical potential energy:

    Pi[u] = integral[
        1/2 E A (u')^2 - E A alpha DeltaT u'
    ] dx - P u(L)

    The objective is divided by a positive constant E*A0*U_SCALE^2/L
    for numerical conditioning. This does not change the minimizer.
    """

    x = x_quad.detach().clone().requires_grad_(True)
    u = model(x)

    du_dx = torch.autograd.grad(
        u,
        x,
        grad_outputs=torch.ones_like(u),
        create_graph=True
    )[0]

    A = area(x)
    dT = temperature_change(x)

    density = (
        0.5 * E * A * du_dx**2
        - E * A * ALPHA * dT * du_dx
    )

    internal_energy = torch.sum(weights * density)

    xL = torch.tensor([[L]], device=DEVICE)
    uL = model(xL)
    external_work = P * uL.squeeze()

    physical_energy = internal_energy - external_work

    scale = E * A0 * U_SCALE**2 / L
    return physical_energy / scale


# ============================================================
# Training
# ============================================================

def train():
    model = DeepRitzThermoelasticity(
        HIDDEN_LAYERS, NEURONS
    ).to(DEVICE)

    xq, wq = gauss_legendre_points(N_QUADRATURE)

    history = []
    phases = []

    t0 = time.perf_counter()

    # ---- Stage 1: Adam ----
    adam = torch.optim.Adam(model.parameters(), lr=ADAM_LR)

    for epoch in range(ADAM_EPOCHS):
        adam.zero_grad()
        loss = compute_energy_loss(model, xq, wq)
        loss.backward()
        adam.step()

        value = float(loss.detach().cpu())
        history.append(value)
        phases.append("Adam")

        if (epoch + 1) % 1000 == 0:
            print(
                f"Adam {epoch+1:5d}/{ADAM_EPOCHS} "
                f"| dimensionless energy = {value:.8e}"
            )

    # ---- Stage 2: L-BFGS ----
    lbfgs = torch.optim.LBFGS(
        model.parameters(),
        max_iter=LBFGS_MAX_ITER,
        tolerance_grad=1e-10,
        tolerance_change=1e-14,
        history_size=100,
        line_search_fn="strong_wolfe"
    )

    calls = 0

    def closure():
        nonlocal calls
        lbfgs.zero_grad()
        loss = compute_energy_loss(model, xq, wq)
        loss.backward()
        history.append(float(loss.detach().cpu()))
        phases.append("L-BFGS")
        calls += 1
        return loss

    lbfgs.step(closure)

    elapsed = time.perf_counter() - t0

    return model, history, phases, elapsed, calls


# ============================================================
# Prediction and validation
# ============================================================

def predict_displacement(model, x):
    xt = torch.tensor(
        x, device=DEVICE
    ).view(-1, 1)

    with torch.no_grad():
        u = model(xt)

    return u.cpu().numpy().ravel()


def predict_stress(model, x):
    xt = torch.tensor(
        x, device=DEVICE
    ).view(-1, 1).requires_grad_(True)

    u = model(xt)

    du_dx = torch.autograd.grad(
        u,
        xt,
        grad_outputs=torch.ones_like(u),
        create_graph=False
    )[0]

    sigma = E * (
        du_dx - ALPHA * temperature_change(xt)
    )

    return sigma.detach().cpu().numpy().ravel()


def metrics(pred, exact):
    e = pred - exact
    return {
        "MAE_m": float(np.mean(np.abs(e))),
        "RMSE_m": float(np.sqrt(np.mean(e**2))),
        "Relative_L2": float(
            np.linalg.norm(e) / np.linalg.norm(exact)
        ),
        "Max_absolute_error_m": float(np.max(np.abs(e))),
    }


def boundary_checks(model):
    x0 = torch.tensor([[0.0]], device=DEVICE)
    xL = torch.tensor([[L]], device=DEVICE).requires_grad_(True)

    u0 = model(x0)
    uL = model(xL)

    duL = torch.autograd.grad(
        uL, xL,
        grad_outputs=torch.ones_like(uL)
    )[0]

    sigmaL = E * (
        duL - ALPHA * temperature_change(xL)
    )

    forceL = area(xL) * sigmaL

    u0 = float(u0.detach().cpu())
    forceL = float(forceL.detach().cpu())

    return {
        "u0_m": u0,
        "target_force_N": P,
        "predicted_force_N": forceL,
        "relative_force_error": abs(forceL - P) / P
    }


# ============================================================
# Save results
# ============================================================

def save_results(model, history, phases, elapsed, lbfgs_calls):

    x = np.linspace(0, L, N_TEST)

    u_exact = analytical_displacement(x)
    u_pred = predict_displacement(model, x)

    sigma_exact = analytical_stress(x)
    sigma_pred = predict_stress(model, x)

    m = metrics(u_pred, u_exact)
    bc = boundary_checks(model)

    # Prediction table
    df = pd.DataFrame({
        "x_m": x,
        "area_m2": area_np(x),
        "temperature_change_K": temperature_change_np(x),
        "u_analytical_m": u_exact,
        "u_deep_ritz_m": u_pred,
        "absolute_error_m": np.abs(u_pred - u_exact),
        "stress_analytical_Pa": sigma_exact,
        "stress_deep_ritz_Pa": sigma_pred
    })

    df.to_csv(
        OUTPUT_DIR / "prediction_data.csv",
        index=False
    )

    all_metrics = {
        **m,
        **bc,
        "training_time_s": elapsed,
        "quadrature_points": N_QUADRATURE,
        "adam_epochs": ADAM_EPOCHS,
        "adam_learning_rate": ADAM_LR,
        "lbfgs_max_iter": LBFGS_MAX_ITER,
        "lbfgs_closure_calls": lbfgs_calls,
        "hidden_layers": HIDDEN_LAYERS,
        "neurons_per_layer": NEURONS,
        "U_SCALE_m": U_SCALE
    }

    pd.DataFrame([all_metrics]).to_csv(
        OUTPUT_DIR / "metrics.csv",
        index=False
    )

    with open(OUTPUT_DIR / "metrics.json", "w") as f:
        json.dump(all_metrics, f, indent=2)

    config = {
        "L_m": L,
        "A0_m2": A0,
        "beta": BETA,
        "E_Pa": E,
        "alpha_1_per_K": ALPHA,
        "Delta_T_max_K": DELTA_T_MAX,
        "P_N": P,
        "hidden_layers": HIDDEN_LAYERS,
        "neurons_per_layer": NEURONS,
        "quadrature_points": N_QUADRATURE,
        "adam_epochs": ADAM_EPOCHS,
        "adam_learning_rate": ADAM_LR,
        "lbfgs_max_iter": LBFGS_MAX_ITER,
        "validation_points": N_TEST,
        "random_seed": SEED
    }

    with open(OUTPUT_DIR / "configuration.json", "w") as f:
        json.dump(config, f, indent=2)

    torch.save(
        model.state_dict(),
        OUTPUT_DIR / "deep_ritz_model.pt"
    )

    # ---- Figures ----
    plt.figure(figsize=(7.5, 5), dpi=300)
    plt.plot(x, area_np(x) * 1e6, linewidth=2.2)
    plt.xlabel("Coordinate x (m)")
    plt.ylabel("Cross-sectional area A(x) (mm$^2$)")
    plt.title("Variable cross-sectional area")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "01_area_profile.png", bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(7.5, 5), dpi=300)
    plt.plot(x, temperature_change_np(x), linewidth=2.2)
    plt.xlabel("Coordinate x (m)")
    plt.ylabel("Temperature increase ΔT(x) (K)")
    plt.title("Prescribed temperature field")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "02_temperature_profile.png", bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(8, 5), dpi=300)
    plt.plot(x, u_exact * 1e3, linewidth=2.5, label="Analytical solution")
    plt.plot(x, u_pred * 1e3, "--", linewidth=2.0, label="Deep Ritz")
    plt.xlabel("Coordinate x (m)")
    plt.ylabel("Displacement u(x) (mm)")
    plt.title("Analytical and Deep Ritz displacement")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "Figure_1_displacement_comparison.png", bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(8, 5), dpi=300)
    plt.plot(x, np.abs(u_pred - u_exact) * 1e6, linewidth=2.0)
    plt.xlabel("Coordinate x (m)")
    plt.ylabel("Absolute error (μm)")
    plt.title("Absolute displacement error on the validation grid")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "Figure_2_absolute_displacement_error.png", bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(8, 5), dpi=300)
    plt.plot(x, sigma_exact / 1e6, linewidth=2.5, label="Analytical solution")
    plt.plot(x, sigma_pred / 1e6, "--", linewidth=2.0, label="Deep Ritz")
    plt.xlabel("Coordinate x (m)")
    plt.ylabel("Axial stress σ(x) (MPa)")
    plt.title("Analytical and Deep Ritz axial stress")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "Figure_3_stress_comparison.png", bbox_inches="tight")
    plt.close()

    adam_count = sum(p == "Adam" for p in phases)
    it = np.arange(1, len(history) + 1)

    plt.figure(figsize=(8, 5), dpi=300)
    plt.semilogy(
        it[:adam_count],
        np.maximum(np.abs(history[:adam_count]), 1e-16),
        linewidth=1.5,
        label="Adam"
    )
    if adam_count < len(history):
        plt.semilogy(
            it[adam_count:],
            np.maximum(np.abs(history[adam_count:]), 1e-16),
            linewidth=1.5,
            label="L-BFGS"
        )
        plt.axvline(
            adam_count,
            linestyle=":",
            linewidth=1.3,
            label="Optimizer switch"
        )
    plt.xlabel("Optimization iteration")
    plt.ylabel("Absolute dimensionless energy")
    plt.title("Deep Ritz energy convergence: Adam and L-BFGS")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "Figure_energy_convergence_Adam_LBFGS.png", bbox_inches="tight")
    plt.close()

    print("\n========== FINAL RESULTS ==========")
    for k, v in all_metrics.items():
        if isinstance(v, (int, float)):
            print(f"{k:28s}: {v:.8e}" if isinstance(v, float) else f"{k:28s}: {v}")

    return all_metrics


def main():
    print("==============================================")
    print("Deep Ritz PINN Thermoelasticity")
    print("Reproducible validation run")
    print("==============================================")
    print("Device:", DEVICE)
    print("Python:", sys.version.split()[0])
    print("Platform:", platform.platform())
    print("PyTorch:", torch.__version__)
    print("NumPy:", np.__version__)
    print("pandas:", pd.__version__)

    model, history, phases, elapsed, calls = train()

    save_results(
        model,
        history,
        phases,
        elapsed,
        calls
    )

    print("\nResults written to:", OUTPUT_DIR.resolve())


if __name__ == "__main__":
    main()


## Automated reproducibility check

The following check distinguishes scientifically equivalent reproduction from platform-dependent last-digit variation.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

results_dir = Path("results")
df = pd.read_csv(results_dir / "prediction_data.csv")
metrics_saved = json.loads((results_dir / "metrics.json").read_text())
exact = df["u_analytical_m"].to_numpy(float)
pred = df["u_deep_ritz_m"].to_numpy(float)
error = pred - exact
exact_stress = df["stress_analytical_Pa"].to_numpy(float)
pred_stress = df["stress_deep_ritz_Pa"].to_numpy(float)

observed = {
    "MAE_m": np.mean(np.abs(error)),
    "RMSE_m": np.sqrt(np.mean(error**2)),
    "Relative_L2_percent": 100*np.linalg.norm(error)/np.linalg.norm(exact),
    "Max_absolute_error_m": np.max(np.abs(error)),
    "Max_stress_relative_error_percent": 100*abs(np.max(pred_stress)-np.max(exact_stress))/np.max(exact_stress),
    "End_force_relative_error_percent": 100*metrics_saved["relative_force_error"],
}
limits = {
    "MAE_m": 3.5e-8,
    "RMSE_m": 4.2e-8,
    "Relative_L2_percent": 0.0135,
    "Max_absolute_error_m": 9.5e-8,
    "Max_stress_relative_error_percent": 2.1,
    "End_force_relative_error_percent": 2.1,
}

for key, value in observed.items():
    status = "PASS" if value <= limits[key] else "FAIL"
    print(f"{key:42s} {value:.12g}  {status}")
assert all(observed[k] <= limits[k] for k in limits), "A reproducibility criterion failed"
print("\nAll numerical acceptance criteria were met.")

## Ten-seed repeatability study
This cell reruns Case A for seeds 0–9 and creates the values used in Table 2.


In [ ]:
# Ten independent initializations for Table 2
repeat_rows = []
for repeat_seed in range(10):
    print(f"Running repeatability seed {repeat_seed}")
    SEED = repeat_seed
    np.random.seed(repeat_seed)
    torch.manual_seed(repeat_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(repeat_seed)
    repeat_model, _, _, repeat_elapsed, repeat_calls = train()
    repeat_x = np.linspace(0.0, L, N_TEST)
    repeat_exact = analytical_displacement(repeat_x)
    repeat_pred = predict_displacement(repeat_model, repeat_x)
    repeat_s_exact = analytical_stress(repeat_x)
    repeat_s_pred = predict_stress(repeat_model, repeat_x)
    repeat_error = repeat_pred - repeat_exact
    repeat_rows.append({
        "seed": repeat_seed,
        "MAE_m": float(np.mean(np.abs(repeat_error))),
        "RMSE_m": float(np.sqrt(np.mean(repeat_error**2))),
        "Relative_L2_percent": float(100*np.linalg.norm(repeat_error)/np.linalg.norm(repeat_exact)),
        "Max_stress_error_percent": float(100*abs(np.max(repeat_s_pred)-np.max(repeat_s_exact))/np.max(repeat_s_exact)),
        "training_time_s": repeat_elapsed,
        "lbfgs_closure_calls": repeat_calls,
    })
repeatability_results = pd.DataFrame(repeat_rows)
repeatability_summary = repeatability_results[[
    "MAE_m", "RMSE_m", "Relative_L2_percent", "Max_stress_error_percent"
]].agg(["mean", "std", "min", "max"])
repeatability_results.to_csv("repeatability_results_v2.csv", index=False)
repeatability_summary.to_csv("repeatability_summary_v2.csv")
display(repeatability_results)
display(repeatability_summary)


## Case B — fixed-fixed rod with restrained thermal expansion
This case creates genuine thermally induced stress. The exact reaction is −60 kN and the analytical stress varies from −60 to −240 MPa.


In [ ]:
"""Fixed-fixed thermoelastic verification case for the Deep Ritz manuscript."""

import json
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

torch.set_default_dtype(torch.float64)
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

L = 1.0
A0 = 1.0e-3
BETA = 3.0
E = 200.0e9
ALPHA = 1.2e-5
DELTA_T_MAX = 100.0
NQ = 200
NVAL = 400
U_SCALE = ALPHA * DELTA_T_MAX * L
OUT = Path("results_fixed_fixed")
OUT.mkdir(exist_ok=True)


def area_t(x):
    return A0 / (1.0 + BETA * (x / L) ** 2)


def dtemp_t(x):
    return DELTA_T_MAX * x / L


def area_np(x):
    return A0 / (1.0 + BETA * (x / L) ** 2)


REACTION = -(
    ALPHA * DELTA_T_MAX * L / 2.0
) / (L / (E * A0) * (1.0 + BETA / 3.0))


def u_exact(x):
    x = np.asarray(x)
    return (
        REACTION / (E * A0) * (x + BETA * x**3 / (3.0 * L**2))
        + ALPHA * DELTA_T_MAX * x**2 / (2.0 * L)
    )


def stress_exact(x):
    return REACTION / area_np(np.asarray(x))


class Model(nn.Module):
    def __init__(self):
        super().__init__()
        layers = [nn.Linear(1, 50), nn.Tanh()]
        for _ in range(3):
            layers.extend([nn.Linear(50, 50), nn.Tanh()])
        layers.append(nn.Linear(50, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        xi = x / L
        return U_SCALE * xi * (1.0 - xi) * self.net(xi)


nodes, weights = np.polynomial.legendre.leggauss(NQ)
xq = torch.tensor(0.5 * L * (nodes + 1.0)).view(-1, 1)
wq = torch.tensor(0.5 * L * weights).view(-1, 1)


def loss_fn(model):
    x = xq.detach().clone().requires_grad_(True)
    u = model(x)
    du = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
    eps_e = du - ALPHA * dtemp_t(x)
    physical = torch.sum(wq * 0.5 * E * area_t(x) * eps_e**2)
    return physical / (E * A0 * U_SCALE**2 / L)


model = Model()
t0 = time.perf_counter()
adam = torch.optim.Adam(model.parameters(), lr=1.0e-3)
for _ in range(5000):
    adam.zero_grad()
    loss = loss_fn(model)
    loss.backward()
    adam.step()

lbfgs = torch.optim.LBFGS(
    model.parameters(), max_iter=500, tolerance_grad=1.0e-10,
    tolerance_change=1.0e-14, history_size=100, line_search_fn="strong_wolfe"
)


def closure():
    lbfgs.zero_grad()
    value = loss_fn(model)
    value.backward()
    return value


lbfgs.step(closure)
elapsed = time.perf_counter() - t0

x = np.linspace(0.0, L, NVAL)
xt = torch.tensor(x).view(-1, 1).requires_grad_(True)
ut = model(xt)
dut = torch.autograd.grad(ut, xt, torch.ones_like(ut))[0]
u_dr = ut.detach().numpy().ravel()
stress_dr = (E * (dut - ALPHA * dtemp_t(xt))).detach().numpy().ravel()
u_ex = u_exact(x)
s_ex = stress_exact(x)

du_err = u_dr - u_ex
rel_l2 = np.linalg.norm(du_err) / np.linalg.norm(u_ex)
max_u_norm = np.max(np.abs(du_err)) / np.max(np.abs(u_ex))
rel_stress_l2 = np.linalg.norm(stress_dr - s_ex) / np.linalg.norm(s_ex)
max_stress_rel = np.max(np.abs(stress_dr - s_ex)) / np.max(np.abs(s_ex))

metrics = {
    "seed": SEED,
    "dtype": "float64",
    "reaction_exact_N": float(REACTION),
    "reaction_left_dr_N": float(stress_dr[0] * area_np(0.0)),
    "reaction_right_dr_N": float(stress_dr[-1] * area_np(L)),
    "relative_L2_displacement_percent": float(100.0 * rel_l2),
    "max_displacement_error_normalized_by_peak_percent": float(100.0 * max_u_norm),
    "relative_L2_stress_percent": float(100.0 * rel_stress_l2),
    "max_stress_error_relative_to_peak_percent": float(100.0 * max_stress_rel),
    "training_time_s": elapsed,
}

pd.DataFrame({
    "x_m": x,
    "area_m2": area_np(x),
    "delta_T_K": DELTA_T_MAX * x / L,
    "u_exact_m": u_ex,
    "u_deep_ritz_m": u_dr,
    "stress_exact_Pa": s_ex,
    "stress_deep_ritz_Pa": stress_dr,
}).to_csv(OUT / "fixed_fixed_fields.csv", index=False)
(OUT / "fixed_fixed_metrics.json").write_text(json.dumps(metrics, indent=2))

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
axes[0].plot(x, 1e6 * u_ex, label="Analytical", lw=2)
axes[0].plot(x, 1e6 * u_dr, "--", label="Deep Ritz", lw=1.7)
axes[0].set(xlabel="x (m)", ylabel="Displacement (µm)")
axes[0].legend()
axes[0].grid(alpha=0.25)
axes[1].plot(x, 1e-6 * s_ex, label="Analytical", lw=2)
axes[1].plot(x, 1e-6 * stress_dr, "--", label="Deep Ritz", lw=1.7)
axes[1].set(xlabel="x (m)", ylabel="Stress (MPa)")
axes[1].legend()
axes[1].grid(alpha=0.25)
fig.tight_layout()
fig.savefig(OUT / "Figure_fixed_fixed_thermoelastic.png", dpi=300)
plt.show()

print(json.dumps(metrics, indent=2))


## Independent finite-element verification
This cell reproduces the FEM convergence values reported in the manuscript.


In [ ]:
# Independent FEM h-convergence check for Case A
def fem_solve(n_elements):
    x_fem = np.linspace(0.0, L, n_elements + 1)
    h = L / n_elements
    K = np.zeros((n_elements + 1, n_elements + 1))
    f = np.zeros(n_elements + 1)
    for e in range(n_elements):
        xm = 0.5 * (x_fem[e] + x_fem[e + 1])
        ae = area_np(xm)
        ke = E * ae / h * np.array([[1.0, -1.0], [-1.0, 1.0]])
        fth = E * ae * ALPHA * temperature_change_np(xm) * np.array([-1.0, 1.0])
        idx = np.array([e, e + 1])
        K[np.ix_(idx, idx)] += ke
        f[idx] += fth
    f[-1] += P
    u_fem = np.zeros(n_elements + 1)
    u_fem[1:] = np.linalg.solve(K[1:, 1:], f[1:])
    return x_fem, u_fem

rows = []
for ne in [50, 100, 200, 400, 2000]:
    xf, uf = fem_solve(ne)
    ue = analytical_displacement(xf)
    err = uf - ue
    rows.append({
        "elements": ne,
        "nodes": ne + 1,
        "relative_L2_error_percent": 100*np.linalg.norm(err)/np.linalg.norm(ue),
        "maximum_absolute_error_m": np.max(np.abs(err)),
    })
fem_table = pd.DataFrame(rows)
fem_table.to_csv("fem_convergence.csv", index=False)
display(fem_table)
assert fem_table.iloc[-1]["relative_L2_error_percent"] < 0.001
print("FEM verification passed: the 2000-element relative L2 error is below 0.001%.")


## Output files
Case A results are saved in `results/`; Case B results in `results_fixed_fixed/`; the FEM table is saved as `fem_convergence.csv`. Use the Files panel in Colab to download them.
